# Data Importing and Understanding

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saicharankomati/dataco-supply-chain-dataset")

print("Path to dataset files:", path)

100%|██████████| 17.8M/17.8M [00:00<00:00, 116MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/saicharankomati/dataco-supply-chain-dataset/versions/1


In [ ]:
import os
import pandas as pd

csv_file = os.path.join(path,"DataCoSupplyChainDataset.csv")

df = pd.read_csv(csv_file,encoding="latin-1") #File uses latin-1 encoding instead of utf-8(regular one)

df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Id',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Email',
 'Customer Fname',
 'Customer Id',
 'Customer Lname',
 'Customer Password',
 'Customer Segment',
 'Customer State',
 'Customer Street',
 'Customer Zipcode',
 'Department Id',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Cardprod Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Order Zipcode',
 'Product Card Id',
 'Product Category Id',
 'Product Description',
 'Product Image',
 'Product Name',
 'Product P

In [ ]:
df.shape

(180519, 53)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer Fname                

In [ ]:
df.dtypes

,0
Type,object
Days for shipping (real),int64
Days for shipment (scheduled),int64
Benefit per order,float64
Sales per customer,float64
Delivery Status,object
Late_delivery_risk,int64
Category Id,int64
Category Name,object
Customer City,object


In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]

,0
Customer Lname,8
Customer Zipcode,3
Order Zipcode,155679
Product Description,180519


# Data Cleaning and Preprocessing

In [ ]:
cols = [
    "Order Id", "order date (DateOrders)", "shipping date (DateOrders)",
    "Days for shipping (real)", "Days for shipment (scheduled)",
    "Delivery Status", "Shipping Mode", "Order Region", "Order Country",
    "Order State", "Category Name", "Sales", "Order Profit Per Order",
    "Order Item Quantity"
]

In [ ]:
df = df[cols]

print(df.shape)
df.columns

(180519, 14)


Index(['Order Id', 'order date (DateOrders)', 'shipping date (DateOrders)',
       'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Delivery Status', 'Shipping Mode', 'Order Region', 'Order Country',
       'Order State', 'Category Name', 'Sales', 'Order Profit Per Order',
       'Order Item Quantity'],
      dtype='object')

In [ ]:
df.columns = [
    "order_id", "order_date", "ship_date", "days_actual",
    "days_scheduled", "delivery_status", "ship_mode", "region",
    "country", "state", "category", "sales", "profit", "quantity"
]

print(df.columns)
df.shape

Index(['order_id', 'order_date', 'ship_date', 'days_actual', 'days_scheduled',
       'delivery_status', 'ship_mode', 'region', 'country', 'state',
       'category', 'sales', 'profit', 'quantity'],
      dtype='object')


(180519, 14)

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

print(df["order_date"].dtype)
print(df["ship_date"].dtype)

datetime64[ns]
datetime64[ns]


In [ ]:
df["delay_days"] = df["days_actual"] - df["days_scheduled"]
df["is_late"] = df["delay_days"] > 0
df["profit_margin"] = (df["profit"] / df["sales"]) * 100
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

In [ ]:
df.dropna(subset=["order_date", "ship_date", "delivery_status"], inplace=True)
df.to_csv("supplychain_clean.csv", index=False)

# EDA

In [ ]:
# Distribution of Delivery Status
print(df["delivery_status"].value_counts())
print()

# Distribution of Shipping mode
print(df["ship_mode"].value_counts())
print()

# On-time rate
on_time_rate = (df["is_late"] == False).sum() / len(df) * 100
print(f"On-time delivery rate: {on_time_rate:.1f}%")
print()

# Average delay
print(f"Avg delay days: {df['delay_days'].mean():.1f}")
print()

# Profit margin summary
print(f"Avg profit margin: {df['profit_margin'].mean():.1f}%")

delivery_status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

ship_mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64

On-time delivery rate: 42.7%

Avg delay days: 0.6

Avg profit margin: 10.8%


In [ ]:
print(df["ship_mode"].value_counts())

ship_mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64


In [ ]:
#Saving it to a csv, easy for downloading
df.to_csv("supplychain_clean.csv", index=False)

# Downloading file
from google.colab import files
files.download("supplychain_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>